In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plyfile import PlyData
from IPython.display import display

# Inputs
folder_path = "/home/nikhita_rayanki/projects/datasets/Elecs"  # folder with your PLY files
output_folder = "/home/nikhita_rayanki/projects/datasets/outputs_Elecs"  # folder to save CSVs and pie charts
os.makedirs(output_folder, exist_ok=True)

# Label map
label_map = {
    "ground": 0,
    "wall": 1,
    "busbar": 2,
    "wire": 3,
    "tower": 4,
    "media": 5,
    "builds": 6,
    "clutter": 7,
    "small": 8,
    "large": 9
}
id_to_name = {v: k for k, v in label_map.items()}
label_range = list(id_to_name.keys())

# Initialize overall counts
overall_counts = {id_to_name[label]: 0 for label in label_range}
overall_total_points = 0
file_stats = []

# Function to create pie chart
def create_pie_chart(sizes, labels_nonzero, title, save_path):
    plt.figure(figsize=(8,8))
    wedges, texts, autotexts = plt.pie(
        sizes, labels=None, autopct=lambda pct: f"{pct:.1f}%", startangle=140
    )
    # Legend: class name + percentage
    legend_labels = [f"{label} - {size:.1f}%" for label, size in zip(labels_nonzero, sizes)]
    plt.legend(wedges, legend_labels, title="Classes", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# Iterate over PLY files
for file_name in os.listdir(folder_path):
    if file_name.endswith(".ply"):
        file_path = os.path.join(folder_path, file_name)
        
        # Read PLY
        plydata = PlyData.read(file_path)
        if 'label' not in plydata['vertex'].data.dtype.names:
            raise ValueError(f"'label' not found in {file_name}")
        
        labels = np.array(plydata['vertex'].data['label'], dtype=int)
        total_points = labels.shape[0]
        
        # Count points per class
        file_counts = {id_to_name[label]: int(np.sum(labels == label)) for label in label_range}
        
        # Densities per class (%)
        file_densities = {f"{id_to_name[label]} (%)": 
                          (file_counts[id_to_name[label]] / total_points) * 100 if total_points > 0 else 0
                          for label in label_range}
        
        # Update overall counts
        for label in label_range:
            overall_counts[id_to_name[label]] += file_counts[id_to_name[label]]
        overall_total_points += total_points
        
        # Save file stats
        file_stats.append({"file": file_name, **file_counts, **file_densities, "total_points": total_points})
        
        # Prepare pie chart for file
        sizes = [file_densities[f"{id_to_name[label]} (%)"] for label in label_range if file_densities[f"{id_to_name[label]} (%)"] > 0]
        labels_nonzero = [id_to_name[label] for label in label_range if file_densities[f"{id_to_name[label]} (%)"] > 0]
        
        # Save pie chart per file
        save_path = os.path.join(output_folder, f"{file_name}_pie.png")
        create_pie_chart(sizes, labels_nonzero, f"Class Densities for {file_name} (%)", save_path)

# Convert per-file stats to DataFrame
file_stats_df = pd.DataFrame(file_stats)

# Convert overall counts to DataFrame
overall_counts_named = {id_to_name[label]: overall_counts[id_to_name[label]] for label in label_range}
overall_densities_named = {f"{id_to_name[label]} (%)": 
                           (overall_counts[id_to_name[label]] / overall_total_points) * 100 if overall_total_points > 0 else 0
                           for label in label_range}
overall_counts_with_density = {**overall_counts_named, **overall_densities_named, "total_points": overall_total_points}
overall_df = pd.DataFrame([overall_counts_with_density], index=["overall"])

# Save CSVs
file_stats_df.to_csv(os.path.join(output_folder, "filewise_class_counts_and_densities_10class.csv"), index=False)
overall_df.to_csv(os.path.join(output_folder, "overall_class_counts_and_densities_10class.csv"))

print(f"✅ CSVs and pie charts (per file + overall) saved in: {output_folder}")

# --------- Overall pie chart for the entire folder ---------
sizes = [overall_densities_named[f"{id_to_name[label]} (%)"] for label in label_range if overall_counts_named[id_to_name[label]] > 0]
labels_nonzero = [id_to_name[label] for label in label_range if overall_counts_named[id_to_name[label]] > 0]

overall_pie_path = os.path.join(output_folder, "overall_class_density_pie.png")
create_pie_chart(sizes, labels_nonzero, "Overall Class Densities (%)", overall_pie_path)

# --------- Display DataFrames in notebook ---------
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1500)

print("===== Filewise Class Counts & Densities =====")
display(file_stats_df.head(10))

print("\n===== Overall Class Counts & Densities =====")
display(overall_df)


✅ CSVs and pie charts (per file + overall) saved in: /home/nikhita_rayanki/projects/datasets/outputs_Elecs
===== Filewise Class Counts & Densities =====


,file,ground,wall,busbar,wire,tower,media,builds,clutter,small,large,ground (%),wall (%),busbar (%),wire (%),tower (%),media (%),builds (%),clutter (%),small (%),large (%),total_points
0,substation4.ply,9271149,1365089,54270,2728451,5130855,6551640,158576,64792,69983,437847,35.889265,5.284355,0.210083,10.562024,19.861898,25.361856,0.613859,0.250814,0.270909,1.694936,25832652
1,substation2.ply,6096583,813662,20688,632907,1289314,798473,298876,33252,55558,2111126,50.175825,6.696565,0.170265,5.208923,10.611254,6.571557,2.459796,0.273669,0.457251,17.374895,12150439
2,substation3.ply,6498633,1176078,129084,1047191,2582343,4819705,209874,24081,31513,1460940,36.144798,6.541237,0.717953,5.824380,14.362754,26.806755,1.167300,0.133936,0.175272,8.125614,17979442
3,substation1.ply,1165659,639563,196764,102113,86339,1792204,43666,4240,5510,261482,27.123866,14.882072,4.578526,2.376080,2.009033,41.703021,1.016070,0.098661,0.128213,6.084458,4297540



===== Overall Class Counts & Densities =====


,ground,wall,busbar,wire,tower,media,builds,clutter,small,large,ground (%),wall (%),busbar (%),wire (%),tower (%),media (%),builds (%),clutter (%),small (%),large (%),total_points
overall,23032024,3994392,400806,4510662,9088851,13962022,710992,126365,162564,4271395,38.221036,6.628588,0.665127,7.485324,15.082708,23.169607,1.179872,0.209699,0.269771,7.088267,60260073
